# 🧬 Dark Manifold V17 — Curriculum Learning

## V16 Failed Because:
- Trained on 5 conditions **simultaneously** → conflicting gradients
- Model confused by different stress responses
- Mean correlation: 0.033 (random guessing)

## V17 Solution: Sequential Training

```
Stage 1: Heat Shock      [500 epochs] → Learn basic stress
            ↓ (keep weights)
Stage 2: Cold Shock      [400 epochs] → Add temperature response  
            ↓ (keep weights)
Stage 3: Oxidative       [400 epochs] → Add redox dynamics
            ↓ (keep weights)
Stage 4: Diauxic Shift   [400 epochs] → Add nutrient sensing
            ↓ (keep weights)
Stage 5: Stationary      [400 epochs] → Add growth arrest
            ↓
TEST: Carbon Variation   [0 training] → UNSEEN!
```

### Key Techniques:
1. **Decreasing Learning Rate** — Protect earlier knowledge
2. **Replay Buffer** — Occasionally revisit old conditions
3. **No Condition Embeddings** — Single unified model

In [ ]:
#@title 1️⃣ Setup
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import time
import json
from copy import deepcopy
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Device: {device}")
if device.type == 'cuda':
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
#@title 2️⃣ Configuration

# Curriculum schedule
STAGE_EPOCHS = [600, 400, 400, 400, 400]  # 2200 total
STAGE_LR = [1e-3, 5e-4, 3e-4, 2e-4, 1e-4]  # Decreasing

# Model
HIDDEN_DIM = 256
N_REACTIONS = 100
DT = 0.02  # Finer integration

# Replay (prevent forgetting)
REPLAY_PROB = 0.2  # 20% chance to replay old condition

# Training order (test condition held out)
TRAIN_ORDER = ['heat_shock', 'cold_shock', 'oxidative_stress', 'diauxic_shift', 'stationary_phase']
TEST_CONDITION = 'carbon_variation'

print("📋 Curriculum Schedule:")
for i, (cond, epochs, lr) in enumerate(zip(TRAIN_ORDER, STAGE_EPOCHS, STAGE_LR)):
    print(f"   Stage {i+1}: {cond:20s} | {epochs} epochs | LR={lr}")
print(f"\n🧪 Test (held out): {TEST_CONDITION}")
print(f"📊 Total training epochs: {sum(STAGE_EPOCHS)}")

In [ ]:
#@title 3️⃣ Create Datasets

def create_datasets():
    metabolites = [
        'glucose-6-P', 'fructose-6-P', 'fructose-1,6-BP', 'DHAP', 'G3P',
        '3-PGA', 'PEP', 'pyruvate',
        'citrate', 'isocitrate', 'alpha-KG', 'succinate', 'fumarate', 'malate', 'oxaloacetate',
        '6-P-gluconate', 'ribose-5-P', 'ribulose-5-P', 'E4P',
        'alanine', 'arginine', 'asparagine', 'aspartate', 'cysteine',
        'glutamate', 'glutamine', 'glycine', 'histidine', 'isoleucine',
        'leucine', 'lysine', 'methionine', 'phenylalanine', 'proline',
        'serine', 'threonine', 'tryptophan', 'tyrosine', 'valine',
        'ATP', 'ADP', 'AMP', 'GTP', 'GDP', 'UTP', 'CTP',
        'NAD', 'NADH', 'NADP', 'NADPH', 'CoA', 'acetyl-CoA',
        'trehalose', 'glycerol', 'lactate', 'acetate', 'formate',
        'cAMP', 'ppGpp', 'homoserine', 'putrescine', 'spermidine'
    ]
    
    n_mets = len(metabolites)
    times = np.array([0, 10, 20, 30, 40, 50, 60, 90, 120, 150, 180]) / 60.0
    aa_names = ['alanine', 'arginine', 'asparagine', 'aspartate', 'cysteine',
                'glutamate', 'glutamine', 'glycine', 'histidine', 'isoleucine',
                'leucine', 'lysine', 'methionine', 'phenylalanine', 'proline',
                'serine', 'threonine', 'tryptophan', 'tyrosine', 'valine']
    aa_indices = [metabolites.index(aa) for aa in aa_names]
    
    def gen_heat():
        data = np.ones((len(times), n_mets))
        for i, met in enumerate(metabolites):
            if met in ['glucose-6-P', 'fructose-6-P', 'fructose-1,6-BP']:
                data[:, i] = 1.0 - 0.5 * (1 - np.exp(-times * 3))
            elif met in aa_names:
                data[:, i] = 1.0 + 0.6 * np.exp(-(times - 0.5)**2 / 0.3)
            elif met == 'trehalose':
                data[:, i] = 1.0 + 2.0 * (1 - np.exp(-times * 1))
            elif met == 'ATP':
                data[:, i] = 1.0 - 0.3 * (1 - np.exp(-times * 4))
        return data
    
    def gen_cold():
        data = np.ones((len(times), n_mets))
        for i, met in enumerate(metabolites):
            if met in ['glucose-6-P', 'fructose-6-P']:
                data[:, i] = 1.0 + 0.4 * (1 - np.exp(-times * 2))
            elif met in aa_names:
                data[:, i] = 1.0 + 0.5 * (1 - np.exp(-times * 1.5))
            elif met in ['ATP', 'GTP']:
                data[:, i] = 1.0 + 0.3 * (1 - np.exp(-times * 1))
        return data
    
    def gen_oxidative():
        data = np.ones((len(times), n_mets))
        for i, met in enumerate(metabolites):
            if met in ['glucose-6-P', '3-PGA']:
                data[:, i] = 1.0 - 0.6 * (1 - np.exp(-times * 5))
            elif met in ['6-P-gluconate', 'ribose-5-P']:
                data[:, i] = 1.0 + 1.2 * (1 - np.exp(-times * 2))
            elif met == 'NADPH':
                data[:, i] = 0.4 + 0.6 * (1 - np.exp(-times * 2))
            elif met in aa_names:
                data[:, i] = 1.0 + 0.2 * np.sin(times * 3) * np.exp(-times * 0.5)
        return data
    
    def gen_diauxic():
        data = np.ones((len(times), n_mets))
        shift = 0.5
        for i, met in enumerate(metabolites):
            if met == 'PEP':
                data[:, i] = 1.0 + 2.5 * np.exp(-(times - shift)**2 / 0.1)
            elif met in ['glucose-6-P', 'fructose-6-P']:
                data[:, i] = np.where(times < shift, 1.0, 0.2 + 0.5 * (times - shift))
            elif met == 'cAMP':
                data[:, i] = 1.0 + 4.0 / (1 + np.exp(-15 * (times - shift)))
            elif met == 'lactate':
                data[:, i] = 0.1 + 1.5 * np.maximum(0, times - shift)
            elif met in aa_names:
                data[:, i] = 1.0 - 0.3 * np.exp(-(times - shift)**2 / 0.2) + 0.15 * times
        return data
    
    def gen_stationary():
        data = np.ones((len(times), n_mets))
        for i, met in enumerate(metabolites):
            if met in ['glucose-6-P', 'fructose-1,6-BP', 'pyruvate']:
                data[:, i] = 1.0 - 0.5 * (1 - np.exp(-times * 0.8))
            elif met in aa_names:
                data[:, i] = 1.0 + 1.2 * (1 - np.exp(-times * 0.5))
            elif met == 'ppGpp':
                data[:, i] = 1.0 + 3.5 * (1 - np.exp(-times * 1))
            elif met == 'trehalose':
                data[:, i] = 1.0 + 2.5 * (1 - np.exp(-times * 0.5))
            elif met == 'ATP':
                data[:, i] = 1.0 - 0.35 * (1 - np.exp(-times * 0.3))
        return data
    
    def gen_carbon():
        data = np.ones((len(times), n_mets))
        for i, met in enumerate(metabolites):
            if met == 'fructose-1,6-BP':
                data[:, i] = 1.8 * np.exp(-times * 1.5) + 0.3
            elif met in ['citrate', 'alpha-KG', 'malate']:
                data[:, i] = 1.0 + 0.8 * (1 - np.exp(-times * 0.8))
            elif met == 'PEP':
                data[:, i] = 1.0 + 0.4 * times
            elif met in aa_names:
                data[:, i] = 1.0 + 0.15 * np.sin(times * 2)
            elif met == 'acetyl-CoA':
                data[:, i] = 1.0 + 1.2 * (1 - np.exp(-times * 0.5))
        return data
    
    datasets = {
        'heat_shock': {'data': gen_heat(), 'times': times, 'metabolites': metabolites, 'aa_indices': aa_indices, 'desc': 'Heat shock'},
        'cold_shock': {'data': gen_cold(), 'times': times, 'metabolites': metabolites, 'aa_indices': aa_indices, 'desc': 'Cold shock'},
        'oxidative_stress': {'data': gen_oxidative(), 'times': times, 'metabolites': metabolites, 'aa_indices': aa_indices, 'desc': 'Oxidative stress'},
        'diauxic_shift': {'data': gen_diauxic(), 'times': times, 'metabolites': metabolites, 'aa_indices': aa_indices, 'desc': 'Diauxic shift'},
        'stationary_phase': {'data': gen_stationary(), 'times': times, 'metabolites': metabolites, 'aa_indices': aa_indices, 'desc': 'Stationary phase'},
        'carbon_variation': {'data': gen_carbon(), 'times': times, 'metabolites': metabolites, 'aa_indices': aa_indices, 'desc': 'Carbon variation'}
    }
    
    for ds in datasets.values():
        ds['data'] = np.clip(ds['data'] + 0.05 * np.random.randn(*ds['data'].shape), 0.05, 10.0)
    
    return datasets, metabolites, aa_indices

all_datasets, metabolite_names, aa_indices = create_datasets()
n_mets = len(metabolite_names)
print(f"✅ {len(all_datasets)} datasets, {n_mets} metabolites, {len(aa_indices)} amino acids")

In [ ]:
#@title 4️⃣ V17 Model (Unified, No Condition Embeddings)

class DarkManifoldV17(nn.Module):
    def __init__(self, n_mets, n_reactions, hidden_dim):
        super().__init__()
        self.n_mets = n_mets
        self.n_reactions = n_reactions
        
        # Stoichiometry (learned metabolic network)
        self.S = nn.Parameter(torch.randn(n_mets, n_reactions) * 0.1)
        
        # Flux network: state → fluxes
        self.flux_net = nn.Sequential(
            nn.Linear(n_mets, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, n_reactions)
        )
        
        # Degradation rates
        self.log_deg = nn.Parameter(torch.zeros(n_mets) - 1.0)
        
        # Regulation network
        self.reg_net = nn.Sequential(
            nn.Linear(n_mets, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, n_mets),
            nn.Tanh()
        )
        
        # Homeostasis baseline
        self.baseline = nn.Parameter(torch.ones(n_mets))
        self.homeo_str = nn.Parameter(torch.tensor(0.05))
    
    def forward(self, M, dt):
        v = self.flux_net(M)
        stoich = torch.matmul(v, self.S.T)
        deg = torch.exp(self.log_deg).clamp(0.01, 0.5) * M
        reg = self.reg_net(M) * M * 0.2
        homeo = self.homeo_str.clamp(0.01, 0.2) * (self.baseline - M)
        
        dM = stoich - deg + reg + homeo
        M_new = torch.clamp(M + dt * dM, 0.01, 15.0)
        return M_new
    
    def simulate(self, M0, times, dt=0.01):
        traj = [M0]
        M = M0.clone()
        t = 0.0
        for target in times[1:]:
            while t < target - 1e-6:
                step = min(dt, target - t)
                M = self.forward(M, step)
                t += step
            traj.append(M.clone())
        return torch.stack(traj, dim=1)

print("✅ V17 model defined")

In [ ]:
#@title 5️⃣ Curriculum Training Function

def train_stage(model, dataset, optimizer, n_epochs, dt, device, 
                replay_datasets=None, replay_prob=0.2):
    """
    Train on one condition, with optional replay of previous conditions.
    """
    model.train()
    losses = []
    
    data = torch.tensor(dataset['data'], dtype=torch.float32, device=device)
    times = torch.tensor(dataset['times'], dtype=torch.float32, device=device)
    
    # Prepare replay data
    replay_data = []
    if replay_datasets:
        for rd in replay_datasets:
            replay_data.append({
                'data': torch.tensor(rd['data'], dtype=torch.float32, device=device),
                'times': torch.tensor(rd['times'], dtype=torch.float32, device=device)
            })
    
    for epoch in range(n_epochs):
        optimizer.zero_grad()
        
        # Decide: train on current or replay old?
        if replay_data and np.random.random() < replay_prob:
            # Replay random old condition
            rd = np.random.choice(replay_data)
            M0 = rd['data'][0:1, :]
            pred = model.simulate(M0, rd['times'], dt=dt).squeeze(0)
            target = rd['data']
        else:
            # Train on current condition
            M0 = data[0:1, :]
            pred = model.simulate(M0, times, dt=dt).squeeze(0)
            target = data
        
        # Multi-scale loss
        loss_mse = F.mse_loss(pred, target)
        loss_log = F.mse_loss(torch.log(pred + 0.01), torch.log(target + 0.01))
        
        # Trajectory smoothness
        diff_pred = pred[1:] - pred[:-1]
        diff_true = target[1:] - target[:-1]
        loss_deriv = F.mse_loss(diff_pred, diff_true)
        
        loss = loss_mse + 0.5 * loss_log + 0.3 * loss_deriv
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        losses.append(loss.item())
    
    return losses

def evaluate(model, dataset, dt, device):
    """Evaluate on a dataset"""
    model.eval()
    with torch.no_grad():
        data = torch.tensor(dataset['data'], dtype=torch.float32, device=device)
        times = torch.tensor(dataset['times'], dtype=torch.float32, device=device)
        
        M0 = data[0:1, :]
        pred = model.simulate(M0, times, dt=dt).squeeze(0)
        
        pred_np = pred.cpu().numpy()
        true_np = data.cpu().numpy()
        
        # Overall correlation
        overall = np.corrcoef(pred_np.flatten(), true_np.flatten())[0, 1]
        
        # Per-metabolite
        met_corrs = {}
        for i, name in enumerate(dataset['metabolites']):
            if np.std(true_np[:, i]) > 0.01:
                c = np.corrcoef(pred_np[:, i], true_np[:, i])[0, 1]
                met_corrs[name] = c if not np.isnan(c) else 0.0
            else:
                met_corrs[name] = 0.0
        
        # AA average
        aa_names = [dataset['metabolites'][i] for i in dataset['aa_indices']]
        aa_corrs = [met_corrs[n] for n in aa_names if n in met_corrs]
        aa_avg = np.mean(aa_corrs) if aa_corrs else 0.0
        
        return {
            'overall': overall,
            'aa_avg': aa_avg,
            'met_corrs': met_corrs,
            'pred': pred_np,
            'true': true_np
        }

print("✅ Training functions defined")

In [ ]:
#@title 6️⃣ Run Curriculum Learning 🚀

print("="*70)
print("CURRICULUM LEARNING — SEQUENTIAL TRAINING")
print("="*70)

# Initialize model
model = DarkManifoldV17(n_mets, N_REACTIONS, HIDDEN_DIM).to(device)
print(f"\n📊 Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Storage
all_losses = []
stage_results = {}
completed_datasets = []

total_start = time.time()

for stage, (condition, epochs, lr) in enumerate(zip(TRAIN_ORDER, STAGE_EPOCHS, STAGE_LR)):
    print(f"\n{'='*70}")
    print(f"STAGE {stage+1}/5: {condition.upper()}")
    print(f"{'='*70}")
    print(f"   Epochs: {epochs} | LR: {lr} | Replay: {len(completed_datasets)} previous conditions")
    
    # Get dataset
    dataset = all_datasets[condition]
    
    # Create optimizer with stage-specific LR
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    
    # Train
    start = time.time()
    losses = train_stage(
        model, dataset, optimizer, epochs, DT, device,
        replay_datasets=completed_datasets if completed_datasets else None,
        replay_prob=REPLAY_PROB
    )
    train_time = time.time() - start
    all_losses.extend(losses)
    
    print(f"   ⏱️ Training time: {train_time/60:.1f} min")
    print(f"   📉 Final loss: {losses[-1]:.4f}")
    
    # Evaluate on current condition
    result = evaluate(model, dataset, DT, device)
    print(f"   🎯 {condition}: corr={result['overall']:.4f}, AA={result['aa_avg']:.4f}")
    
    # Evaluate on ALL previous conditions (check for forgetting)
    if completed_datasets:
        print(f"\n   📊 Checking retention on previous conditions:")
        for prev_cond in [d['name'] for d in completed_datasets]:
            prev_result = evaluate(model, all_datasets[prev_cond], DT, device)
            status = "✅" if prev_result['overall'] > 0.5 else "⚠️" if prev_result['overall'] > 0.2 else "❌"
            print(f"      {status} {prev_cond}: {prev_result['overall']:.4f}")
    
    # Store results
    stage_results[condition] = {
        'overall': result['overall'],
        'aa_avg': result['aa_avg'],
        'train_time': train_time
    }
    
    # Add to completed (for replay)
    completed_datasets.append({'name': condition, **dataset})

total_time = time.time() - total_start
print(f"\n{'='*70}")
print(f"CURRICULUM COMPLETE — Total time: {total_time/60:.1f} min")
print(f"{'='*70}")

In [ ]:
#@title 7️⃣ TEST on Held-Out Condition 🧪

print("\n" + "="*70)
print(f"TESTING ON HELD-OUT CONDITION: {TEST_CONDITION.upper()}")
print("="*70)
print("\n⚠️ This condition was NEVER seen during training!\n")

test_dataset = all_datasets[TEST_CONDITION]
test_result = evaluate(model, test_dataset, DT, device)

print(f"🎯 Overall Correlation: {test_result['overall']:.4f}")
print(f"🧬 Amino Acid Average:  {test_result['aa_avg']:.4f}")

# Breakdown
sorted_mets = sorted(test_result['met_corrs'].items(), key=lambda x: x[1], reverse=True)
excellent = sum(1 for _, c in sorted_mets if c > 0.8)
good = sum(1 for _, c in sorted_mets if 0.5 < c <= 0.8)
moderate = sum(1 for _, c in sorted_mets if 0.2 < c <= 0.5)
poor = sum(1 for _, c in sorted_mets if 0 < c <= 0.2)
negative = sum(1 for _, c in sorted_mets if c <= 0)

print(f"\n📊 Metabolite Breakdown ({len(sorted_mets)} total):")
print(f"   ✅ Excellent (r>0.8): {excellent}")
print(f"   ✅ Good (0.5-0.8):    {good}")
print(f"   ⚠️ Moderate (0.2-0.5): {moderate}")
print(f"   ❌ Poor (0-0.2):       {poor}")
print(f"   ⛔ Negative:          {negative}")

# Compare to V16
print(f"\n{'='*70}")
print("V16 vs V17 COMPARISON")
print(f"{'='*70}")
print(f"""
┌────────────────────────────────────────────────────────────────────┐
│                    GENERALIZATION TEST RESULTS                      │
├────────────────────────────────────────────────────────────────────┤
│                                                                    │
│   V16 (Simultaneous):     0.033 overall,  -0.28 AA average        │
│   V17 (Curriculum):       {test_result['overall']:.3f} overall,   {test_result['aa_avg']:.2f} AA average        │
│                                                                    │
│   Improvement:            {'+' if test_result['overall'] > 0.033 else ''}{(test_result['overall'] - 0.033):.3f}                                     │
│                                                                    │
└────────────────────────────────────────────────────────────────────┘
""")

if test_result['overall'] > 0.5:
    print("🏆 SUCCESS! Curriculum learning enables generalization!")
elif test_result['overall'] > 0.3:
    print("✅ IMPROVEMENT! Better than random, but room to grow.")
elif test_result['overall'] > 0.1:
    print("⚠️ MARGINAL. Some learning, but still struggling.")
else:
    print("❌ Still failing to generalize. Need different approach.")

In [ ]:
#@title 8️⃣ Final Evaluation on ALL Conditions

print("\n" + "="*70)
print("FINAL EVALUATION ON ALL CONDITIONS")
print("="*70)

final_results = {}

print(f"\n{'Condition':<25} {'Overall':>12} {'AA Avg':>12} {'Status':>10}")
print("-" * 60)

for cond_name, dataset in all_datasets.items():
    result = evaluate(model, dataset, DT, device)
    final_results[cond_name] = result
    
    is_test = "🧪 TEST" if cond_name == TEST_CONDITION else ""
    status = "✅" if result['overall'] > 0.5 else "⚠️" if result['overall'] > 0.2 else "❌"
    print(f"{status} {cond_name:<22} {result['overall']:>12.4f} {result['aa_avg']:>12.4f} {is_test:>10}")

# Summary
train_corrs = [final_results[c]['overall'] for c in TRAIN_ORDER]
test_corr = final_results[TEST_CONDITION]['overall']

print("-" * 60)
print(f"{'TRAIN MEAN':<25} {np.mean(train_corrs):>12.4f}")
print(f"{'TEST (held-out)':<25} {test_corr:>12.4f}")
print(f"{'GENERALIZATION GAP':<25} {np.mean(train_corrs) - test_corr:>12.4f}")

In [ ]:
#@title 9️⃣ Visualize Results

fig = plt.figure(figsize=(16, 12))

# 1. Training curve
ax1 = fig.add_subplot(2, 2, 1)
ax1.semilogy(all_losses, alpha=0.7)
# Mark stage boundaries
cumsum = 0
for i, epochs in enumerate(STAGE_EPOCHS):
    cumsum += epochs
    ax1.axvline(x=cumsum, color='red', linestyle='--', alpha=0.5)
    ax1.text(cumsum - epochs/2, max(all_losses)*0.5, TRAIN_ORDER[i].replace('_', '\n'), 
             ha='center', fontsize=8)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss (log)')
ax1.set_title('Curriculum Training Loss', fontsize=12)
ax1.grid(True, alpha=0.3)

# 2. Bar chart of all conditions
ax2 = fig.add_subplot(2, 2, 2)
conds = list(final_results.keys())
corrs = [final_results[c]['overall'] for c in conds]
colors = ['blue' if c != TEST_CONDITION else 'red' for c in conds]
bars = ax2.bar(range(len(conds)), corrs, color=colors, edgecolor='black', alpha=0.7)
ax2.set_xticks(range(len(conds)))
ax2.set_xticklabels([c.replace('_', '\n') for c in conds], fontsize=9)
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax2.axhline(y=np.mean(train_corrs), color='blue', linestyle='--', label=f'Train mean: {np.mean(train_corrs):.3f}')
ax2.set_ylabel('Correlation')
ax2.set_title(f'Performance by Condition (Red = Held-Out Test)', fontsize=12)
ax2.legend()
ax2.set_ylim(-0.5, 1.0)
ax2.grid(axis='y', alpha=0.3)

# 3. Test condition trajectories
ax3 = fig.add_subplot(2, 2, 3)
test_pred = final_results[TEST_CONDITION]['pred']
test_true = final_results[TEST_CONDITION]['true']
times = all_datasets[TEST_CONDITION]['times'] * 60
plot_mets = ['glucose-6-P', 'pyruvate', 'glutamate', 'ATP', 'citrate']
for met in plot_mets:
    if met in metabolite_names:
        idx = metabolite_names.index(met)
        ax3.plot(times, test_true[:, idx], 'o-', label=f'{met} (true)', lw=2, ms=4)
        ax3.plot(times, test_pred[:, idx], 's--', label=f'{met} (pred)', lw=1.5, ms=3, alpha=0.7)
ax3.set_xlabel('Time (min)')
ax3.set_ylabel('Concentration')
ax3.set_title(f'TEST: {TEST_CONDITION} (r={test_corr:.3f})', fontsize=12)
ax3.legend(fontsize=7, ncol=2)
ax3.grid(True, alpha=0.3)

# 4. Scatter plot
ax4 = fig.add_subplot(2, 2, 4)
ax4.scatter(test_true.flatten(), test_pred.flatten(), alpha=0.3, s=15)
max_val = max(test_true.max(), test_pred.max())
ax4.plot([0, max_val], [0, max_val], 'r--', lw=2)
ax4.set_xlabel('True')
ax4.set_ylabel('Predicted')
ax4.set_title(f'Test: Predicted vs True (r={test_corr:.3f})', fontsize=12)
ax4.grid(True, alpha=0.3)

plt.suptitle(f'Dark Manifold V17 — Curriculum Learning\nTest Corr: {test_corr:.4f} (V16: 0.033)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('v17_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n💾 Saved: v17_results.png")

In [ ]:
#@title 🔟 Save Results

save_dict = {
    'version': 'V17',
    'method': 'Curriculum Learning',
    'train_order': TRAIN_ORDER,
    'test_condition': TEST_CONDITION,
    'stage_epochs': STAGE_EPOCHS,
    'stage_lr': STAGE_LR,
    'test_overall_corr': float(test_corr),
    'test_aa_avg': float(final_results[TEST_CONDITION]['aa_avg']),
    'train_mean_corr': float(np.mean(train_corrs)),
    'generalization_gap': float(np.mean(train_corrs) - test_corr),
    'all_results': {k: {'overall': float(v['overall']), 'aa_avg': float(v['aa_avg'])} 
                   for k, v in final_results.items()},
    'v16_comparison': {
        'v16_test_corr': 0.033,
        'v17_test_corr': float(test_corr),
        'improvement': float(test_corr - 0.033)
    },
    'total_epochs': sum(STAGE_EPOCHS),
    'metabolite_names': metabolite_names
}

with open('v17_results.json', 'w') as f:
    json.dump(save_dict, f, indent=2)
print("💾 Saved: v17_results.json")

torch.save({
    'model_state_dict': model.state_dict(),
    'config': {'n_mets': n_mets, 'n_reactions': N_REACTIONS, 'hidden_dim': HIDDEN_DIM},
    'test_corr': test_corr,
    'train_order': TRAIN_ORDER
}, 'v17_model.pt')
print("💾 Saved: v17_model.pt")

# Download
from google.colab import files
files.download('v17_results.json')
files.download('v17_results.png')
files.download('v17_model.pt')

# 📌 V17 Summary

## What Changed from V16

| Aspect | V16 (Failed) | V17 (Curriculum) |
|--------|--------------|------------------|
| Training | All 5 conditions simultaneously | Sequential, one at a time |
| Condition info | Learned embeddings | None (unified model) |
| Learning rate | Fixed | Decreasing per stage |
| Forgetting prevention | None | Replay buffer (20%) |
| Gradient conflicts | Severe | Minimized |

## Why Curriculum Learning Should Help

1. **No conflicting gradients** — Model focuses on one condition at a time
2. **Knowledge accumulation** — Each stage builds on previous
3. **Replay prevents forgetting** — Occasionally revisit old conditions
4. **Decreasing LR** — Protect early learning from being overwritten

## Next Steps If V17 Works
- Try different curriculum orders
- Test on multiple held-out conditions
- Scale to more conditions

## Next Steps If V17 Fails
- Need fundamentally different approach:
  - Meta-learning (MAML)
  - Hierarchical models
  - Physics-informed constraints